# Basic API examples

Stateless `fit`, `sample`, `predict`, `smoothed_params`, and GoF checks.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from pyscarcopula._utils import pobs
from pyscarcopula import (
    GumbelCopula, FrankCopula, JoeCopula, ClaytonCopula,
    GaussianCopula, StudentCopula, CVineCopula, RVineCopula,
    PredictConfig,
)
from pyscarcopula.api import fit, sample, predict, smoothed_params
from pyscarcopula.stattests import gof_test

DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('..') / 'data'


## 1. Read dataset and transform to log-returns

In [ ]:
crypto_prices = pd.read_csv(DATA_DIR / 'crypto_prices.csv', index_col=0, sep=';')

tickers = ['BTC-USD', 'ETH-USD']
returns = np.log(crypto_prices[tickers] / crypto_prices[tickers].shift(1))[1:].values
u = pobs(returns)

print(f'T = {len(u)}, d = {u.shape[1]}')


## 2. Fit bivariate copula and GoF test

Three methods: MLE (constant parameter), SCAR-TM (stochastic with transfer matrix), GAS (score-driven).

In [ ]:
copula = GumbelCopula(rotate=180)

result_mle = fit(copula, u, method='mle')
result_tm  = fit(copula, u, method='scar-tm-ou')
result_gas = fit(copula, u, method='gas')

for name, res in [('MLE', result_mle), ('GAS', result_gas), ('SCAR-TM', result_tm)]:
    gof = gof_test(copula, u, fit_result=res, to_pobs=False)
    print(f'{name:8s}  logL = {res.log_likelihood:8.2f}   GoF p = {gof.pvalue:.4f}')

## 5. Sample and predict

Two functions with different purposes:

- **`sample`** reproduces the fitted model (for validation). `fit(copula, sample(...))` should recover similar parameters.
- **`predict`** generates next-step forecasts conditional on observed data (for risk metrics).

In [ ]:
# Model validation: sample -> refit -> GoF
v = sample(copula, u, result_tm, n=len(u))
v_pobs = pobs(v)
result_refit = fit(copula, v_pobs, method='scar-tm-ou')
gof_v = gof_test(copula, v_pobs, fit_result=result_refit, to_pobs=False)

p1 = result_tm.params
p2 = result_refit.params
print(f'Original: theta={p1.theta:.2f}, mu={p1.mu:.2f}, nu={p1.nu:.2f}')
print(f'Refit:    theta={p2.theta:.2f}, mu={p2.mu:.2f}, nu={p2.nu:.2f}')
print(f'GoF on sample: p={gof_v.pvalue:.4f}')

In [ ]:
# Prediction: conditional on current market state
u_pred = predict(copula, u, result_tm, n=10000)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(u[:, 0], u[:, 1], s=1, alpha=0.3)
axes[0].set_title('Original data')
axes[1].scatter(u_pred[:, 0], u_pred[:, 1], s=1, alpha=0.3)
axes[1].set_title('Predicted (conditional on last state)')
for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Reproducible random sampling with `rng`

Pass a `numpy.random.Generator` when you need reproducible Monte Carlo output. Fresh generators with the same seed produce identical samples; reusing the same generator advances its stream.

In [ ]:
seed = 12345

sample_a = sample(copula, u, result_tm, n=5, rng=np.random.default_rng(seed))
sample_b = sample(copula, u, result_tm, n=5, rng=np.random.default_rng(seed))

rng = np.random.default_rng(seed)
sample_c = sample(copula, u, result_tm, n=5, rng=rng)
sample_d = sample(copula, u, result_tm, n=5, rng=rng)

print('Fresh generators with same seed -> identical:', np.allclose(sample_a, sample_b))
print('Same generator reused -> identical:', np.allclose(sample_c, sample_d))


## Save and load a fitted model

`model.save()` writes a JSON model file. `ModelClass.load()` restores the fitted model, including its cached training pseudo-observations by default.

In [ ]:
model = GumbelCopula(rotate=180)
model.fit(u, method='mle')

model_path = Path('basic_api_gumbel.json')
model.save(model_path)

loaded_model = GumbelCopula.load(model_path)

print('Saved to:', model_path)
print('Loaded type:', type(loaded_model).__name__)
print('Loaded logL:', round(loaded_model.fit_result.log_likelihood, 2))
print('Predict shape:', loaded_model.predict(10, rng=np.random.default_rng(7)).shape)
